# Volatility Signal Model — VN100

VOLATILITY SIGNAL MODEL — VN100 | Monthly rebalancing | 3 portfolios

Purpose
-------
Build sector-neutral, market-cap-weighted P1/P2/P3 portfolios using volatility/risk signals.

Strategy structure
------------------
1. Universe: VN100 stocks available in the uploaded price, sector, and market-cap data.
2. Signals, computed using trailing 60 trading days:
   - Standard deviation of daily returns
   - Maximum drawdown
   - Beta versus VNINDEX
3. Ranking:
   - Within each sector, sort stocks from low risk to high risk.
   - P1 = lowest-risk third
   - P2 = middle third
   - P3 = highest-risk third
4. Weighting:
   - Each portfolio replicates the sector weights of the eligible VN100 universe.
   - Within each sector bucket, stocks are weighted by market cap.
5. Rebalancing:
   - Monthly, using the last available trading date of each month.
6. Performance:
   - Gross and net returns
   - Turnover and fee drag
   - P1/P2/P3 comparison
   - P1 minus P3 spread
   - Benchmark comparison versus VNINDEX
   - Cumulative returns
   - CSV / Excel / ZIP output

This script is designed to follow the data-cleaning, sector-tertile assignment,
market-cap weighting, turnover, fee-drag, and performance-testing logic used in
your existing value/momentum codebase.

In [ ]:
# ── CELL 1: IMPORT LIBRARIES ─────────────────────────────────────────

import os
import io
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    files = None

print("✓ Libraries loaded")


In [ ]:
# ── CELL 2: CONFIGURATION ────────────────────────────────────────────
# Edit this cell to match your data files.

# Price file column names
COL_DATE    = "time"
COL_TICKER  = "ticker"
COL_CLOSE   = "close"
COL_VOLUME  = "volume"
PRICE_DATE_FORMAT = "%m/%d/%Y %H:%M"

# Sector file / valuation file column names
SECTOR_TICKER_COL = "Ticker"
SECTOR_COL        = "Industrial sector (ICB) L1"

# Market-cap file column names
MCAP_COL_TICKER = "ticker"
MCAP_COL_DATE   = "Date"
MCAP_COL_MCAP   = "Market cap"
MCAP_DATE_FORMAT = "%m/%d/%Y"

# VNINDEX file column names
# Used for beta signal and benchmark comparison.
VN_DATE_COL     = "date"
VN_CLOSE_COL    = "price"
VN_DATE_FORMAT  = "%m/%d/%Y"

# Volatility signal settings
VOL_WINDOW       = 60
MIN_OBS_SIGNAL   = 45
TRADING_DAYS_ANNUAL = 252

# Portfolio settings
PORTFOLIOS       = ["P1", "P2", "P3"]
MIN_SECTOR_SIZE  = 6      # sectors below this size go to P2, matching your old logic
MIN_UNIVERSE_SIZE = 50    # skip rebalance months with too few eligible stocks

# Optional manual VN100 ticker filter.
# If your price files contain only VN100 names, leave this as None.
# If your price files contain more than VN100 names, paste the current VN100 tickers here.
VN100_TICKERS = None
# Example:
# VN100_TICKERS = ["ACB", "BCM", "BID", "BVH", ...]

# Trading fee settings
BUY_FEE_PCT  = 0.15
SELL_FEE_PCT = 0.15
SELL_TAX_PCT = 0.10

# Output settings
OUTPUT_DIR = "volatility_model_outputs"

print("✓ Configuration set")
print(f"  Volatility window : {VOL_WINDOW} trading days")
print(f"  Min signal obs    : {MIN_OBS_SIGNAL}")
print(f"  Rebalancing       : Monthly")
print(f"  Portfolios        : P1/P2/P3, low risk → high risk")
print(f"  Weighting         : VN100 sector weight × market-cap weight within sector bucket")
print(f"  Round-trip cost   : {BUY_FEE_PCT + SELL_FEE_PCT + SELL_TAX_PCT:.2f}%")


In [ ]:
# ── CELL 3: HELPER FUNCTIONS — FILE I/O AND DATE PARSING ─────────────

def _maybe_download(path):
    """Download a file in Colab; otherwise just print the local path."""
    if IN_COLAB:
        files.download(path)
    else:
        print(f"Saved locally: {path}")


def upload_files(prompt):
    """Colab upload helper."""
    if not IN_COLAB:
        raise RuntimeError(
            "This script is currently configured for Colab upload. "
            "If running locally, replace files.upload() with local file paths."
        )
    print(prompt)
    uploaded = files.upload()
    print(f"\n✓ {len(uploaded)} file(s) uploaded:")
    for fname in uploaded:
        print(f"  {fname}")
    return uploaded


def read_uploaded_table(uploaded_dict, filename):
    """Read CSV or Excel file from a Colab uploaded-file dictionary."""
    content = uploaded_dict[filename]
    lower = filename.lower()

    if lower.endswith(".xlsx") or lower.endswith(".xls"):
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(io.BytesIO(content))


def parse_date_series(s, date_format=None, dayfirst_fallback=False):
    """
    Robust date parser.
    First tries the explicit format.
    If many dates fail, falls back to pandas inference.
    """

    if date_format is not None:
        parsed = pd.to_datetime(s, format=date_format, errors="coerce")
    else:
        parsed = pd.to_datetime(s, errors="coerce")

    # Fallback for old / inconsistent files
    if parsed.notna().mean() < 0.95:
        fallback = pd.to_datetime(
            s,
            errors="coerce",
            dayfirst=dayfirst_fallback
        )
        parsed = parsed.fillna(fallback)

    return parsed


def ensure_upper_ticker(s):
    return s.astype(str).str.strip().str.upper()


def month_end_trading_dates(daily_index):
    """
    Return the actual last trading date of each calendar month.
    This avoids using non-trading calendar month-end dates.
    """
    idx = pd.DatetimeIndex(daily_index).sort_values()
    idx = idx[~idx.duplicated()]
    s = pd.Series(idx, index=idx)
    return pd.DatetimeIndex(s.groupby(idx.to_period("M")).max()).sort_values()


print("✓ File and date helpers loaded")


In [ ]:
# ── CELL 4: UPLOAD AND LOAD PRICE DATA ───────────────────────────────

uploaded_prices = upload_files("Upload all VN100 stock price CSV files...")

print("=" * 80)
print("LOADING PRICE DATA")
print("=" * 80)

price_frames = []

for fname, content in uploaded_prices.items():
    df = pd.read_csv(io.BytesIO(content))
    print(f"  {fname}: {len(df):>10,} rows | {df[COL_TICKER].nunique():>4} tickers")
    price_frames.append(df)

prices_raw = pd.concat(price_frames, ignore_index=True)

# Standardize column names
rename_map = {
    COL_DATE: "date",
    COL_TICKER: "ticker",
    COL_CLOSE: "close",
}

if COL_VOLUME in prices_raw.columns:
    rename_map[COL_VOLUME] = "volume"

prices_raw = prices_raw.rename(columns=rename_map)

required_price_cols = ["date", "ticker", "close"]
missing_price_cols = [c for c in required_price_cols if c not in prices_raw.columns]
if missing_price_cols:
    raise ValueError(f"Price data missing required columns: {missing_price_cols}")

# Clean price data
prices_raw["date"]   = parse_date_series(prices_raw["date"], PRICE_DATE_FORMAT)
prices_raw["ticker"] = ensure_upper_ticker(prices_raw["ticker"])
prices_raw["close"]  = pd.to_numeric(prices_raw["close"], errors="coerce")

if "volume" in prices_raw.columns:
    prices_raw["volume"] = pd.to_numeric(prices_raw["volume"], errors="coerce")

prices_raw = (
    prices_raw
    .dropna(subset=["date", "ticker", "close"])
    .drop_duplicates(subset=["date", "ticker"], keep="last")
    .query("close > 0")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

if VN100_TICKERS is not None:
    vn100_set = set([str(t).strip().upper() for t in VN100_TICKERS])
    prices_raw = prices_raw[prices_raw["ticker"].isin(vn100_set)].copy()

daily_wide = (
    prices_raw
    .pivot_table(index="date", columns="ticker", values="close", aggfunc="last")
    .sort_index()
)

daily_returns = daily_wide.pct_change().replace([np.inf, -np.inf], np.nan)

print(f"\n  Combined rows : {len(prices_raw):,}")
print(f"  Unique tickers: {prices_raw['ticker'].nunique()}")
print(f"  Date range    : {prices_raw['date'].min().date()} → {prices_raw['date'].max().date()}")
print(f"  Daily matrix  : {daily_wide.shape}")
print("✓ Price data loaded")


In [ ]:
# ── CELL 5: UPLOAD AND LOAD SECTOR DATA ──────────────────────────────
# Use your old valuation/fundamental file if it contains Ticker and sector.

uploaded_sector = upload_files("Upload the sector / valuation file containing ticker and sector columns...")

sector_filename = list(uploaded_sector.keys())[0]
sector_raw = read_uploaded_table(uploaded_sector, sector_filename)

sector_raw.columns = [str(c).replace("\n", " ").strip() for c in sector_raw.columns]

required_sector_cols = [SECTOR_TICKER_COL, SECTOR_COL]
missing_sector_cols = [c for c in required_sector_cols if c not in sector_raw.columns]

if missing_sector_cols:
    raise ValueError(f"Sector file missing required columns: {missing_sector_cols}")

sector_df = sector_raw[[SECTOR_TICKER_COL, SECTOR_COL]].copy()
sector_df = sector_df.rename(columns={
    SECTOR_TICKER_COL: "ticker",
    SECTOR_COL: "sector",
})

sector_df["ticker"] = ensure_upper_ticker(sector_df["ticker"])
sector_df["sector"] = sector_df["sector"].astype(str).str.strip()

sector_df = (
    sector_df
    .dropna(subset=["ticker", "sector"])
    .drop_duplicates(subset=["ticker"], keep="last")
    .sort_values("ticker")
    .reset_index(drop=True)
)

sector_live = sector_df.set_index("ticker")["sector"].to_dict()

print("=" * 80)
print("SECTOR DATA")
print("=" * 80)
print(f"  File          : {sector_filename}")
print(f"  Tickers       : {sector_df['ticker'].nunique()}")
print(f"  Sectors       : {sector_df['sector'].nunique()}")
print("\nSector breakdown:")
print(sector_df["sector"].value_counts().to_string())
print("✓ Sector data loaded")


In [ ]:
# ── CELL 6: UPLOAD AND LOAD MARKET-CAP DATA ──────────────────────────

uploaded_mcap = upload_files("Upload your market-cap file(s), CSV or Excel...")

print("=" * 80)
print("LOADING MARKET CAP DATA")
print("=" * 80)

mcap_frames = []

for fname in uploaded_mcap:
    df = read_uploaded_table(uploaded_mcap, fname)
    print(f"  {fname}: raw shape {df.shape}")
    mcap_frames.append(df)

mcap_raw = pd.concat(mcap_frames, ignore_index=True)

mcap_raw = mcap_raw.rename(columns={
    MCAP_COL_TICKER: "ticker",
    MCAP_COL_DATE: "date",
    MCAP_COL_MCAP: "mcap",
})

required_mcap_cols = ["ticker", "date", "mcap"]
missing_mcap_cols = [c for c in required_mcap_cols if c not in mcap_raw.columns]
if missing_mcap_cols:
    raise ValueError(f"Market-cap data missing required columns: {missing_mcap_cols}")

mcap_raw["ticker"] = ensure_upper_ticker(mcap_raw["ticker"])
mcap_raw["date"]   = parse_date_series(mcap_raw["date"], MCAP_DATE_FORMAT)
mcap_raw["mcap"]   = pd.to_numeric(mcap_raw["mcap"], errors="coerce")

mcap_raw = (
    mcap_raw
    .dropna(subset=["ticker", "date", "mcap"])
    .drop_duplicates(subset=["ticker", "date"], keep="last")
    .query("mcap > 0")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

if VN100_TICKERS is not None:
    mcap_raw = mcap_raw[mcap_raw["ticker"].isin(vn100_set)].copy()

mcap_wide = (
    mcap_raw
    .pivot_table(index="date", columns="ticker", values="mcap", aggfunc="last")
    .sort_index()
)

print(f"\n  Combined rows : {len(mcap_raw):,}")
print(f"  Unique tickers: {mcap_raw['ticker'].nunique()}")
print(f"  Date range    : {mcap_raw['date'].min().date()} → {mcap_raw['date'].max().date()}")
print(f"  Mcap matrix   : {mcap_wide.shape}")
print("✓ Market cap data loaded")


In [ ]:
# ── CELL 7: UPLOAD AND LOAD VNINDEX DATA ─────────────────────────────
# Used for:
#   1. Beta signal
#   2. Benchmark comparison

uploaded_vnindex = upload_files("Upload VNINDEX daily price file...")

vnindex_filename = list(uploaded_vnindex.keys())[0]
vn_raw = read_uploaded_table(uploaded_vnindex, vnindex_filename)

vn_raw.columns = [str(c).strip() for c in vn_raw.columns]

required_vn_cols = [VN_DATE_COL, VN_CLOSE_COL]
missing_vn_cols = [c for c in required_vn_cols if c not in vn_raw.columns]
if missing_vn_cols:
    raise ValueError(f"VNINDEX file missing required columns: {missing_vn_cols}")

vn_raw = vn_raw.rename(columns={
    VN_DATE_COL: "date",
    VN_CLOSE_COL: "close",
})

vn_raw["date"]  = parse_date_series(vn_raw["date"], VN_DATE_FORMAT)
vn_raw["close"] = pd.to_numeric(vn_raw["close"], errors="coerce")

vn_raw = (
    vn_raw
    .dropna(subset=["date", "close"])
    .drop_duplicates(subset=["date"], keep="last")
    .query("close > 0")
    .sort_values("date")
    .reset_index(drop=True)
)

vnindex_px = vn_raw.set_index("date")["close"].sort_index()

print("=" * 80)
print("VNINDEX DATA")
print("=" * 80)
print(f"  File       : {vnindex_filename}")
print(f"  Obs        : {len(vnindex_px):,}")
print(f"  Date range : {vnindex_px.index.min().date()} → {vnindex_px.index.max().date()}")
print("✓ VNINDEX data loaded")


In [ ]:
# ── CELL 8: PORTFOLIO ASSIGNMENT AND WEIGHTING HELPERS ───────────────

def assign_within_sector_tertiles(
    df,
    signal_col,
    sector_col,
    ticker_col="ticker",
    min_sector_size=MIN_SECTOR_SIZE,
    higher_is_better=True,
):
    """
    Assign stocks to P1, P2, P3 based on within-sector signal rank.

    For each sector:
      - Best floor(n/3) stocks    -> P1
      - Worst floor(n/3) stocks   -> P3
      - Remaining stocks          -> P2
      - Sectors below min size     -> P2

    For volatility signals, use higher_is_better=False:
      low volatility / low drawdown / low beta = P1.
    """

    result = df.copy()
    result["portfolio"]          = "P2"
    result["within_sector_rank"] = np.nan
    result["within_sector_size"] = np.nan

    for sector, group in result.groupby(sector_col):
        valid_idx = group[signal_col].dropna().index
        n = len(valid_idx)

        result.loc[group.index, "within_sector_size"] = n

        if n < min_sector_size:
            continue

        ranked = result.loc[valid_idx, signal_col].rank(
            ascending=not higher_is_better,
            method="first"
        )

        result.loc[valid_idx, "within_sector_rank"] = ranked

        n_top = int(np.floor(n / 3))
        n_bot = int(np.floor(n / 3))

        for idx, rank in ranked.items():
            if rank <= n_top:
                result.loc[idx, "portfolio"] = "P1"
            elif rank > n - n_bot:
                result.loc[idx, "portfolio"] = "P3"
            else:
                result.loc[idx, "portfolio"] = "P2"

    return result


def compute_sector_replicated_mcap_weights(assigned_df, portfolio_name):
    """
    Compute market-cap weights for one portfolio.

    Weighting method:
      1. Compute sector weights from the full eligible universe.
      2. Keep only sectors represented in the target portfolio.
      3. Redistribute unavailable sector weights across represented sectors.
      4. Within each represented sector, weight stocks by market cap.

    Final stock weight:
      weight_i = target_sector_weight × stock_mcap / portfolio_sector_mcap
    """

    eligible = assigned_df.copy()
    eligible = eligible.dropna(subset=["ticker", "sector", "mcap"])
    eligible = eligible[eligible["mcap"] > 0]

    pf_df = eligible[eligible["portfolio"] == portfolio_name].copy()

    if pf_df.empty:
        return pd.DataFrame()

    # Sector weights from full eligible universe
    universe_sector_mcap = eligible.groupby("sector")["mcap"].sum()
    universe_total_mcap = universe_sector_mcap.sum()

    if universe_total_mcap <= 0:
        return pd.DataFrame()

    sector_weight_universe = universe_sector_mcap / universe_total_mcap

    # Market cap inside target portfolio by sector
    pf_sector_mcap = pf_df.groupby("sector")["mcap"].sum()
    represented_sectors = pf_sector_mcap[pf_sector_mcap > 0].index

    sector_weight_target = sector_weight_universe.reindex(
        represented_sectors
    ).fillna(0)

    if sector_weight_target.sum() <= 0:
        return pd.DataFrame()

    # Redistribute only across sectors represented in this portfolio
    sector_weight_target = sector_weight_target / sector_weight_target.sum()

    pf_df["sector_weight_target"] = pf_df["sector"].map(sector_weight_target)
    pf_df["portfolio_sector_mcap"] = pf_df["sector"].map(pf_sector_mcap)

    pf_df["weight_within_sector"] = (
        pf_df["mcap"] / pf_df["portfolio_sector_mcap"]
    )

    pf_df["weight"] = (
        pf_df["sector_weight_target"] *
        pf_df["weight_within_sector"]
    )

    pf_df["weight"] = (
        pf_df["weight"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    if pf_df["weight"].sum() > 0:
        pf_df["weight"] = pf_df["weight"] / pf_df["weight"].sum()
    else:
        return pd.DataFrame()

    return pf_df


def print_portfolio_sizes(df, date_col="date", label=""):
    sizes = df.groupby([date_col, "portfolio"]).size().unstack(fill_value=0)
    avg = sizes.mean()
    print(f"\n{label} average portfolio sizes:")
    print(f"  P1: {avg.get('P1', 0):.0f} stocks | "
          f"P2: {avg.get('P2', 0):.0f} stocks | "
          f"P3: {avg.get('P3', 0):.0f} stocks")


print("✓ Portfolio helpers loaded")


In [ ]:
# ── CELL 9: COMPUTE VOLATILITY SIGNAL MATRICES ───────────────────────

print("=" * 80)
print("COMPUTING VOLATILITY SIGNALS")
print("=" * 80)


def rolling_max_drawdown_matrix(price_df, window=VOL_WINDOW, min_periods=MIN_OBS_SIGNAL):
    """
    Rolling maximum drawdown using trailing price window.
    Returns positive drawdown values:
      0.20 means -20% max drawdown.
    """

    def _mdd(x):
        x = np.asarray(x, dtype=float)
        x = x[~np.isnan(x)]
        if len(x) < min_periods:
            return np.nan
        if np.any(x <= 0):
            x = x[x > 0]
        if len(x) < min_periods:
            return np.nan
        peak = np.maximum.accumulate(x)
        dd = x / peak - 1.0
        return -np.min(dd)

    return price_df.rolling(window=window, min_periods=min_periods).apply(
        _mdd,
        raw=True
    )


# 1. Standard deviation of daily returns
std_60d = (
    daily_returns
    .rolling(window=VOL_WINDOW, min_periods=MIN_OBS_SIGNAL)
    .std()
    * np.sqrt(TRADING_DAYS_ANNUAL)
)

# 2. Maximum drawdown of trailing price path
mdd_60d = rolling_max_drawdown_matrix(
    daily_wide,
    window=VOL_WINDOW,
    min_periods=MIN_OBS_SIGNAL
)

# 3. Beta versus VNINDEX
vnindex_aligned_px = vnindex_px.reindex(daily_wide.index.union(vnindex_px.index)).sort_index().ffill()
vnindex_aligned_px = vnindex_aligned_px.reindex(daily_wide.index).ffill()
vnindex_daily_ret = vnindex_aligned_px.pct_change().replace([np.inf, -np.inf], np.nan)

mkt_var_60d = vnindex_daily_ret.rolling(
    window=VOL_WINDOW,
    min_periods=MIN_OBS_SIGNAL
).var()

beta_60d = (
    daily_returns
    .rolling(window=VOL_WINDOW, min_periods=MIN_OBS_SIGNAL)
    .cov(vnindex_daily_ret)
    .divide(mkt_var_60d, axis=0)
)

signal_matrices = {
    "std_60d": std_60d,
    "mdd_60d": mdd_60d,
    "beta_60d": beta_60d,
}

for name, mat in signal_matrices.items():
    latest_count = mat.iloc[-1].replace([np.inf, -np.inf], np.nan).dropna().shape[0]
    print(f"  {name:<8}: matrix {mat.shape}, latest valid signals = {latest_count}")

print("✓ Volatility signals computed")


In [ ]:
# ── CELL 10: PREPARE MONTHLY REBALANCE DATES AND ALIGNED DATA ────────

rebalance_dates = month_end_trading_dates(daily_wide.index)

# Only keep dates where signal can exist
first_signal_date = daily_wide.index[min(VOL_WINDOW, len(daily_wide.index) - 1)]
rebalance_dates = rebalance_dates[rebalance_dates >= first_signal_date]

# Price matrix at rebalance dates
price_at_rebal = daily_wide.reindex(rebalance_dates, method="ffill")

# Market cap at rebalance dates
mcap_date_marker = pd.DataFrame(
    np.repeat(mcap_wide.index.to_numpy().reshape(-1, 1), len(mcap_wide.columns), axis=1),
    index=mcap_wide.index,
    columns=mcap_wide.columns
).where(mcap_wide.notna())

all_mcap_dates = mcap_wide.index.union(rebalance_dates).sort_values()

mcap_at_rebal = (
    mcap_wide
    .reindex(all_mcap_dates)
    .ffill()
    .reindex(rebalance_dates)
)

mcap_date_at_rebal = (
    mcap_date_marker
    .reindex(all_mcap_dates)
    .ffill()
    .reindex(rebalance_dates)
)


def get_period_returns(start_date, end_date):
    """
    Return from start_date to end_date using prices available at rebalance dates.
    """

    if start_date not in price_at_rebal.index or end_date not in price_at_rebal.index:
        return pd.Series(dtype=float)

    start_px = price_at_rebal.loc[start_date]
    end_px = price_at_rebal.loc[end_date]

    ret = end_px / start_px - 1
    ret = ret.replace([np.inf, -np.inf], np.nan)

    return ret


print("=" * 80)
print("MONTHLY REBALANCE SETUP")
print("=" * 80)
print(f"  Rebalance months : {len(rebalance_dates)}")
print(f"  Date range       : {rebalance_dates.min().date()} → {rebalance_dates.max().date()}")
print(f"  Latest mcap coverage: {mcap_at_rebal.iloc[-1].notna().sum()} stocks")
print("✓ Monthly data aligned")


In [ ]:
# ── CELL 11: RUN MONTHLY VOLATILITY-SIGNAL BACKTEST ──────────────────

print("=" * 80)
print("RUNNING MONTHLY VOLATILITY-SIGNAL BACKTEST")
print("=" * 80)

all_result_rows = []
all_stock_rows = []

for signal_name, signal_matrix in signal_matrices.items():

    print(f"\nSignal: {signal_name}")

    prev_weights_by_pf = {pf: {} for pf in PORTFOLIOS}
    prev_date_by_pf = {pf: None for pf in PORTFOLIOS}

    for i, date in enumerate(rebalance_dates):

        if date not in signal_matrix.index:
            continue
        if date not in mcap_at_rebal.index:
            continue

        signal_row = (
            signal_matrix
            .loc[date]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if VN100_TICKERS is not None:
            signal_row = signal_row[signal_row.index.isin(vn100_set)]

        if len(signal_row) < MIN_UNIVERSE_SIZE:
            continue

        mcap_row = mcap_at_rebal.loc[date]
        mcap_date_row = mcap_date_at_rebal.loc[date]

        month_df = pd.DataFrame({
            "ticker": signal_row.index,
            "risk_signal": signal_row.values,
            "sector": [sector_live.get(t, np.nan) for t in signal_row.index],
            "mcap": [mcap_row.get(t, np.nan) for t in signal_row.index],
            "mcap_date": [mcap_date_row.get(t, pd.NaT) for t in signal_row.index],
        })

        month_df = (
            month_df
            .dropna(subset=["ticker", "sector", "risk_signal", "mcap"])
            .query("mcap > 0")
            .copy()
        )

        if len(month_df) < MIN_UNIVERSE_SIZE:
            continue

        # For all volatility/risk signals:
        # lower signal = lower risk = P1.
        assigned = assign_within_sector_tertiles(
            df=month_df,
            signal_col="risk_signal",
            sector_col="sector",
            ticker_col="ticker",
            min_sector_size=MIN_SECTOR_SIZE,
            higher_is_better=False,
        )

        assigned["signal_name"] = signal_name

        # Holding-period return from current rebalance date to next rebalance date
        if i < len(rebalance_dates) - 1:
            next_date = rebalance_dates[i + 1]
            ret_current_to_next = get_period_returns(date, next_date)
        else:
            next_date = pd.NaT
            ret_current_to_next = None

        for pf in PORTFOLIOS:

            pf_df = compute_sector_replicated_mcap_weights(
                assigned_df=assigned,
                portfolio_name=pf
            )

            if pf_df.empty:
                continue

            curr_weights = dict(zip(pf_df["ticker"], pf_df["weight"]))
            curr_tickers = set(curr_weights.keys())

            prev_weights = prev_weights_by_pf.get(pf, {})
            prev_tickers = set(prev_weights.keys())
            prev_date = prev_date_by_pf.get(pf)

            total_mcap = pf_df["mcap"].sum()

            # Fee and turnover calculation
            if prev_date is not None and len(prev_weights) > 0:

                entering = curr_tickers - prev_tickers
                exiting = prev_tickers - curr_tickers
                holding = curr_tickers & prev_tickers

                ret_prev_to_current = get_period_returns(prev_date, date)

                drifted_values = {}

                for t, prev_w in prev_weights.items():
                    r = ret_prev_to_current.get(t, 0.0)
                    if pd.isna(r):
                        r = 0.0
                    drifted_values[t] = prev_w * (1.0 + r)

                total_drifted_value = sum(drifted_values.values())

                if total_drifted_value > 0:
                    drifted_weights = {
                        t: v / total_drifted_value
                        for t, v in drifted_values.items()
                    }
                else:
                    drifted_weights = {
                        t: 0.0
                        for t in prev_weights.keys()
                    }

                # Entering stocks: buy current target weight
                buy_cost = sum(
                    curr_weights[t] * (BUY_FEE_PCT / 100.0)
                    for t in entering
                )

                # Exiting stocks: sell drifted pre-rebalance weight
                sell_cost = sum(
                    drifted_weights.get(t, 0.0) *
                    ((SELL_FEE_PCT + SELL_TAX_PCT) / 100.0)
                    for t in exiting
                )

                # Holding stocks: cost only on active rebalance trade
                rebal_cost = 0.0

                for t in holding:
                    target_w = curr_weights.get(t, 0.0)
                    drifted_w = drifted_weights.get(t, 0.0)

                    active_trade_w = target_w - drifted_w

                    if active_trade_w > 0:
                        rebal_cost += active_trade_w * (BUY_FEE_PCT / 100.0)
                    elif active_trade_w < 0:
                        rebal_cost += abs(active_trade_w) * (
                            (SELL_FEE_PCT + SELL_TAX_PCT) / 100.0
                        )

                fee_drag = buy_cost + sell_cost + rebal_cost

                entering_turnover = sum(curr_weights[t] for t in entering)
                exiting_turnover = sum(drifted_weights.get(t, 0.0) for t in exiting)

                holding_turnover = 0.0
                for t in holding:
                    target_w = curr_weights.get(t, 0.0)
                    drifted_w = drifted_weights.get(t, 0.0)
                    holding_turnover += abs(target_w - drifted_w)

                turnover = (
                    entering_turnover +
                    exiting_turnover +
                    holding_turnover
                ) / 2.0

            else:
                # First rebalance: buy the full portfolio
                entering = curr_tickers
                exiting = set()

                buy_cost = sum(
                    curr_weights[t] * (BUY_FEE_PCT / 100.0)
                    for t in curr_tickers
                )
                sell_cost = 0.0
                rebal_cost = 0.0
                fee_drag = buy_cost
                turnover = 1.0

            # Holding-period return
            if ret_current_to_next is not None:
                weighted_ret = 0.0
                used_weight = 0.0

                for ticker, weight in curr_weights.items():
                    r = ret_current_to_next.get(ticker, np.nan)

                    if not pd.isna(r):
                        weighted_ret += weight * r
                        used_weight += weight

                gross_ret = weighted_ret / used_weight if used_weight > 0 else np.nan
                net_ret = gross_ret - fee_drag if not pd.isna(gross_ret) else np.nan

            else:
                gross_ret = np.nan
                net_ret = np.nan

            # Save portfolio-level result
            all_result_rows.append({
                "signal_name": signal_name,
                "date": date,
                "period_end": next_date,
                "portfolio": pf,
                "n_stocks": len(curr_tickers),
                "n_entering": len(entering),
                "n_exiting": len(exiting),
                "turnover": turnover,
                "buy_cost": buy_cost,
                "sell_cost": sell_cost,
                "rebal_cost": rebal_cost,
                "fee_drag": fee_drag,
                "gross_ret": gross_ret,
                "net_ret": net_ret,
                "total_mcap": total_mcap,
                "eligible_universe_size": len(month_df),
            })

            # Save stock-level assignments
            pf_df["signal_name"] = signal_name
            pf_df["date"] = date
            pf_df["period_end"] = next_date
            pf_df["status"] = pf_df["ticker"].apply(
                lambda t: "NEW" if t in entering else "HOLD"
            )

            all_stock_rows.append(
                pf_df[[
                    "signal_name", "date", "period_end",
                    "ticker", "sector",
                    "portfolio", "risk_signal",
                    "within_sector_rank", "within_sector_size",
                    "mcap", "mcap_date",
                    "sector_weight_target",
                    "weight_within_sector",
                    "weight",
                    "status"
                ]]
            )

            # Update previous weights
            prev_weights_by_pf[pf] = curr_weights
            prev_date_by_pf[pf] = date


vol_results = pd.DataFrame(all_result_rows)

if not all_stock_rows:
    raise RuntimeError(
        "No stock-level portfolio rows were generated. "
        "Check price, sector, market-cap coverage, and date settings."
    )

vol_stock_assign = pd.concat(all_stock_rows, ignore_index=True)

vol_stock_assign = vol_stock_assign.sort_values(
    ["signal_name", "date", "portfolio", "weight"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

print("\n✓ Monthly volatility-signal portfolios built")
print(f"  Signals computed: {vol_results['signal_name'].nunique()}")
print(f"  Months computed : {vol_results['date'].nunique()}")

print("\nAverage portfolio size by signal:")
print(
    vol_results
    .groupby(["signal_name", "portfolio"])["n_stocks"]
    .mean()
    .round(1)
    .unstack()
    .to_string()
)

print("\nAverage monthly turnover by signal:")
print(
    vol_results
    .groupby(["signal_name", "portfolio"])["turnover"]
    .mean()
    .unstack()
    .applymap(lambda x: f"{x:.1%}")
    .to_string()
)


In [ ]:
# ── CELL 12: LATEST PORTFOLIO PREVIEW ────────────────────────────────

def fmt_pct(x, decimals=2):
    if pd.isna(x):
        return "-"
    return f"{x:.{decimals}%}"


def fmt_float(x, decimals=4):
    if pd.isna(x):
        return "-"
    return f"{x:.{decimals}f}"


def fmt_date(x):
    if pd.isna(x):
        return "-"
    return str(pd.to_datetime(x).date())


latest_date = vol_stock_assign["date"].max()
latest_assign = vol_stock_assign[vol_stock_assign["date"] == latest_date].copy()

print("=" * 100)
print(f"LATEST VOLATILITY MODEL HOLDINGS as of {latest_date.date()}")
print("=" * 100)

for signal_name in sorted(latest_assign["signal_name"].unique()):
    print(f"\nSIGNAL: {signal_name}")

    signal_latest = latest_assign[latest_assign["signal_name"] == signal_name]

    for pf, label in [
        ("P1", "LOW RISK"),
        ("P2", "MIDDLE"),
        ("P3", "HIGH RISK"),
    ]:
        pf_df = (
            signal_latest[signal_latest["portfolio"] == pf]
            .sort_values("weight", ascending=False)
        )

        print(f"\n{'─' * 100}")
        print(f"{pf}: {label} | {len(pf_df)} stocks")
        print(f"{'─' * 100}")

        print(
            f"{'Ticker':<8} "
            f"{'Sector':<24} "
            f"{'Weight':>8} "
            f"{'Mcap (B)':>12} "
            f"{'Mcap Date':<12} "
            f"{'Risk Sig':>12} "
            f"{'Rank':>6} "
            f"{'Status':<6}"
        )

        for _, r in pf_df.head(20).iterrows():
            print(
                f"{r['ticker']:<8} "
                f"{str(r['sector']):<24} "
                f"{r['weight']:>7.2%} "
                f"{r['mcap']/1e9:>11.0f} "
                f"{fmt_date(r['mcap_date']):<12} "
                f"{fmt_float(r['risk_signal'], 4):>12} "
                f"{fmt_float(r['within_sector_rank'], 0):>6} "
                f"{str(r['status']):<6}"
            )

        if len(pf_df) > 20:
            print(f"... and {len(pf_df) - 20} more stocks")

        print(f"\nTotal weight : {pf_df['weight'].sum():.2%}")
        print(f"Top 5 weight : {pf_df['weight'].head(5).sum():.1%}")
        print(f"Top 10 weight: {pf_df['weight'].head(10).sum():.1%}")


In [ ]:
# ── CELL 13: PERFORMANCE SUMMARY ─────────────────────────────────────

def max_drawdown(return_series):
    r = (
        pd.Series(return_series)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    if len(r) == 0:
        return np.nan

    wealth = (1.0 + r).cumprod()
    running_max = wealth.cummax()
    drawdown = wealth / running_max - 1.0

    return drawdown.min()


def performance_summary(results_df, annual_factor=12):
    """
    Performance summary by signal and portfolio.
    """

    rows = []

    for (signal_name, pf), g in results_df.groupby(["signal_name", "portfolio"]):
        g = g.sort_values("date").copy()

        common = (
            g[["gross_ret", "net_ret", "turnover", "fee_drag", "n_stocks"]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["gross_ret", "net_ret"])
        )

        if common.empty:
            continue

        gross = common["gross_ret"]
        net = common["net_ret"]

        ann_gross = gross.mean() * annual_factor
        ann_net = net.mean() * annual_factor

        cagr_net = (1 + net).prod() ** (annual_factor / len(net)) - 1
        ann_vol = net.std(ddof=1) * np.sqrt(annual_factor)
        sharpe = ann_net / ann_vol if ann_vol > 0 else np.nan

        rows.append({
            "signal_name": signal_name,
            "portfolio": pf,
            "n_periods": len(common),
            "avg_n_stocks": common["n_stocks"].mean(),
            "ann_gross_return": ann_gross,
            "ann_net_return": ann_net,
            "cagr_net": cagr_net,
            "ann_volatility": ann_vol,
            "sharpe_zero_rf": sharpe,
            "max_drawdown": max_drawdown(net),
            "avg_monthly_turnover": common["turnover"].mean(),
            "ann_fee_drag": common["fee_drag"].mean() * annual_factor,
            "win_rate_positive": (net > 0).mean(),
        })

    return pd.DataFrame(rows)


def p1_minus_p3_spread(results_df):
    """
    Build P1-P3 spread return table by signal.
    """

    pieces = []

    for signal_name, g in results_df.groupby("signal_name"):
        wide = g.pivot_table(
            index=["date", "period_end"],
            columns="portfolio",
            values=["gross_ret", "net_ret"],
            aggfunc="first"
        )

        if ("net_ret", "P1") not in wide.columns or ("net_ret", "P3") not in wide.columns:
            continue

        spread = pd.DataFrame(index=wide.index).reset_index()
        spread["signal_name"] = signal_name
        spread["gross_p1_minus_p3"] = wide[("gross_ret", "P1")].values - wide[("gross_ret", "P3")].values
        spread["net_p1_minus_p3"] = wide[("net_ret", "P1")].values - wide[("net_ret", "P3")].values

        pieces.append(spread)

    if pieces:
        spread_df = pd.concat(pieces, ignore_index=True)
    else:
        spread_df = pd.DataFrame()

    return spread_df


def spread_summary(spread_df, annual_factor=12):
    rows = []

    for signal_name, g in spread_df.groupby("signal_name"):
        x = (
            g["net_p1_minus_p3"]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            .astype(float)
        )

        if len(x) < 2:
            continue

        mean_monthly = x.mean()
        std_monthly = x.std(ddof=1)
        t_stat = mean_monthly / (std_monthly / np.sqrt(len(x))) if std_monthly > 0 else np.nan

        rows.append({
            "signal_name": signal_name,
            "n_periods": len(x),
            "ann_net_p1_minus_p3": mean_monthly * annual_factor,
            "monthly_mean": mean_monthly,
            "monthly_vol": std_monthly,
            "t_stat_monthly_mean": t_stat,
            "spread_hit_rate": (x > 0).mean(),
        })

    return pd.DataFrame(rows)


vol_perf_summary = performance_summary(vol_results, annual_factor=12)
vol_spread = p1_minus_p3_spread(vol_results)
vol_spread_summary = spread_summary(vol_spread, annual_factor=12)

print("=" * 100)
print("VOLATILITY MODEL PERFORMANCE SUMMARY")
print("=" * 100)

display_cols = [
    "signal_name", "portfolio", "n_periods", "avg_n_stocks",
    "ann_gross_return", "ann_net_return", "cagr_net",
    "ann_volatility", "sharpe_zero_rf", "max_drawdown",
    "avg_monthly_turnover", "ann_fee_drag", "win_rate_positive"
]

print(
    vol_perf_summary[display_cols]
    .sort_values(["signal_name", "portfolio"])
    .to_string(index=False)
)

print("\nP1 MINUS P3 SPREAD SUMMARY")
print(vol_spread_summary.to_string(index=False))


In [ ]:
# ── CELL 14: BENCHMARK COMPARISON VS VNINDEX ─────────────────────────

def compute_vnindex_period_returns(results_df, vnindex_px):
    """
    Compute VNINDEX return from date to period_end for each period.
    Uses latest available VNINDEX close on or before each date.
    """

    periods = (
        results_df[["date", "period_end"]]
        .drop_duplicates()
        .dropna()
        .copy()
    )

    periods["date"] = pd.to_datetime(periods["date"])
    periods["period_end"] = pd.to_datetime(periods["period_end"])

    needed_dates = (
        vnindex_px.index
        .union(pd.DatetimeIndex(periods["date"]))
        .union(pd.DatetimeIndex(periods["period_end"]))
        .sort_values()
    )

    vn_ffill = vnindex_px.reindex(needed_dates).ffill()

    rows = []

    for _, r in periods.iterrows():
        start = r["date"]
        end = r["period_end"]

        if start not in vn_ffill.index or end not in vn_ffill.index:
            vn_ret = np.nan
            start_px = np.nan
            end_px = np.nan
        else:
            start_px = vn_ffill.loc[start]
            end_px = vn_ffill.loc[end]

            if pd.isna(start_px) or pd.isna(end_px) or start_px <= 0:
                vn_ret = np.nan
            else:
                vn_ret = end_px / start_px - 1.0

        rows.append({
            "date": start,
            "period_end": end,
            "vnindex_ret": vn_ret,
            "vnindex_start_close": start_px,
            "vnindex_end_close": end_px,
        })

    return pd.DataFrame(rows)


def attach_vnindex_returns(results_df, vnindex_px):
    df = results_df.copy()
    df["date"] = pd.to_datetime(df["date"])
    df["period_end"] = pd.to_datetime(df["period_end"])

    vn_period = compute_vnindex_period_returns(df, vnindex_px)

    out = df.merge(
        vn_period,
        on=["date", "period_end"],
        how="left"
    )

    out["active_gross_ret"] = out["gross_ret"] - out["vnindex_ret"]
    out["active_net_ret"] = out["net_ret"] - out["vnindex_ret"]

    return out


def ols_alpha_beta(model_ret, benchmark_ret, annual_factor=12):
    """
    Estimate model_ret = alpha + beta * benchmark_ret + error.
    Returns annualized alpha, beta, alpha t-stat.
    """

    data = pd.concat([
        pd.Series(model_ret).rename("model"),
        pd.Series(benchmark_ret).rename("benchmark")
    ], axis=1).replace([np.inf, -np.inf], np.nan).dropna()

    if len(data) < 4:
        return np.nan, np.nan, np.nan

    y = data["model"].values
    x = data["benchmark"].values

    if np.std(x, ddof=1) == 0:
        return np.nan, np.nan, np.nan

    X = np.column_stack([np.ones(len(x)), x])

    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta_hat

    dof = len(y) - X.shape[1]
    if dof <= 0:
        return np.nan, np.nan, np.nan

    sigma2 = (resid @ resid) / dof

    try:
        cov_beta = sigma2 * np.linalg.inv(X.T @ X)
        alpha_se = np.sqrt(cov_beta[0, 0])
    except np.linalg.LinAlgError:
        return np.nan, np.nan, np.nan

    alpha_periodic = beta_hat[0]
    beta = beta_hat[1]

    alpha_annual = alpha_periodic * annual_factor
    alpha_t = alpha_periodic / alpha_se if alpha_se > 0 else np.nan

    return alpha_annual, beta, alpha_t


def performance_vs_vnindex(df, annual_factor=12):
    """
    Build performance summary vs VNINDEX by signal and portfolio.
    """

    rows = []

    for (signal_name, pf), g in df.groupby(["signal_name", "portfolio"]):
        g = g.sort_values("date").copy()

        common = (
            g[["net_ret", "gross_ret", "vnindex_ret", "turnover", "fee_drag"]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        if common.empty:
            continue

        net = common["net_ret"]
        gross = common["gross_ret"]
        bench = common["vnindex_ret"]
        active = net - bench

        ann_net = net.mean() * annual_factor
        ann_gross = gross.mean() * annual_factor
        ann_bench = bench.mean() * annual_factor
        ann_active = active.mean() * annual_factor

        cagr_net = (1 + net).prod() ** (annual_factor / len(net)) - 1
        cagr_bench = (1 + bench).prod() ** (annual_factor / len(bench)) - 1

        ann_vol = net.std(ddof=1) * np.sqrt(annual_factor)
        ann_bench_vol = bench.std(ddof=1) * np.sqrt(annual_factor)
        tracking_error = active.std(ddof=1) * np.sqrt(annual_factor)

        sharpe = ann_net / ann_vol if ann_vol > 0 else np.nan
        info_ratio = ann_active / tracking_error if tracking_error > 0 else np.nan

        alpha, beta, alpha_t = ols_alpha_beta(
            model_ret=net,
            benchmark_ret=bench,
            annual_factor=annual_factor
        )

        rows.append({
            "signal_name": signal_name,
            "portfolio": pf,
            "n_periods": len(common),
            "ann_gross_return": ann_gross,
            "ann_net_return": ann_net,
            "vnindex_ann_return": ann_bench,
            "ann_active_return": ann_active,
            "model_cagr": cagr_net,
            "vnindex_cagr": cagr_bench,
            "ann_volatility": ann_vol,
            "vnindex_ann_volatility": ann_bench_vol,
            "tracking_error": tracking_error,
            "sharpe_zero_rf": sharpe,
            "information_ratio": info_ratio,
            "alpha_vs_vnindex": alpha,
            "beta_vs_vnindex": beta,
            "alpha_t_stat": alpha_t,
            "max_drawdown": max_drawdown(net),
            "vnindex_max_drawdown": max_drawdown(bench),
            "win_rate_vs_vnindex": (active > 0).mean(),
            "avg_turnover": g["turnover"].mean(),
            "ann_fee_drag": g["fee_drag"].mean() * annual_factor,
        })

    return pd.DataFrame(rows)


vol_vs_vnindex = attach_vnindex_returns(vol_results, vnindex_px)
vol_benchmark_summary = performance_vs_vnindex(vol_vs_vnindex, annual_factor=12)

print("=" * 100)
print("PERFORMANCE VS VNINDEX")
print("=" * 100)
print(
    vol_benchmark_summary
    .sort_values(["signal_name", "portfolio"])
    .to_string(index=False)
)


In [ ]:
# ── CELL 15: CUMULATIVE RETURN TABLES AND CHARTS ─────────────────────

def cumulative_return_table(df):
    pieces = []

    for (signal_name, pf), g in df.groupby(["signal_name", "portfolio"]):
        g = (
            g
            .sort_values("date")
            .replace([np.inf, -np.inf], np.nan)
            .dropna(subset=["net_ret", "vnindex_ret"])
            .copy()
        )

        if g.empty:
            continue

        g["model_cum_net"] = (1.0 + g["net_ret"]).cumprod() - 1.0
        g["vnindex_cum"] = (1.0 + g["vnindex_ret"]).cumprod() - 1.0
        g["active_cum"] = (1.0 + (g["net_ret"] - g["vnindex_ret"])).cumprod() - 1.0

        pieces.append(
            g[[
                "signal_name", "portfolio", "date", "period_end",
                "net_ret", "gross_ret", "vnindex_ret",
                "model_cum_net", "vnindex_cum", "active_cum"
            ]]
        )

    if pieces:
        return pd.concat(pieces, ignore_index=True)

    return pd.DataFrame()


vol_cumulative = cumulative_return_table(vol_vs_vnindex)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save one cumulative chart per signal/portfolio
for (signal_name, pf), plot_df in vol_cumulative.groupby(["signal_name", "portfolio"]):
    plot_df = plot_df.sort_values("date")

    if plot_df.empty:
        continue

    plt.figure(figsize=(10, 5))
    plt.plot(plot_df["date"], plot_df["model_cum_net"], label=f"{signal_name} {pf} net")
    plt.plot(plot_df["date"], plot_df["vnindex_cum"], label="VNINDEX")
    plt.axhline(0, linewidth=0.8)
    plt.title(f"{signal_name} {pf} Net Cumulative Return vs VNINDEX")
    plt.xlabel("Date")
    plt.ylabel("Cumulative Return")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    chart_name = f"cum_return_{signal_name}_{pf.lower()}_vs_vnindex.png"
    chart_path = os.path.join(OUTPUT_DIR, chart_name)

    plt.savefig(chart_path, dpi=150)
    plt.close()

print("✓ Cumulative return tables and charts created")


In [ ]:
# ── CELL 16: SAVE OUTPUT FILES ───────────────────────────────────────

print("=" * 100)
print("SAVING VOLATILITY MODEL OUTPUTS")
print("=" * 100)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Main CSV outputs
paths = {}

paths["results"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_monthly_market_cap_weighted_results.csv"
)
vol_results.to_csv(paths["results"], index=False)

paths["stock_assignments"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_monthly_market_cap_weighted_stock_assignments.csv"
)
vol_stock_assign.to_csv(paths["stock_assignments"], index=False)

paths["performance_summary"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_performance_summary.csv"
)
vol_perf_summary.to_csv(paths["performance_summary"], index=False)

paths["p1_minus_p3_spreads"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_p1_minus_p3_spreads.csv"
)
vol_spread.to_csv(paths["p1_minus_p3_spreads"], index=False)

paths["p1_minus_p3_summary"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_p1_minus_p3_summary.csv"
)
vol_spread_summary.to_csv(paths["p1_minus_p3_summary"], index=False)

paths["vs_vnindex_returns"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_vs_vnindex_period_returns.csv"
)
vol_vs_vnindex.to_csv(paths["vs_vnindex_returns"], index=False)

paths["vs_vnindex_summary"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_vs_vnindex_summary.csv"
)
vol_benchmark_summary.to_csv(paths["vs_vnindex_summary"], index=False)

paths["cumulative"] = os.path.join(
    OUTPUT_DIR,
    "volatility_model_cumulative_returns.csv"
)
vol_cumulative.to_csv(paths["cumulative"], index=False)

# Per-signal/per-portfolio stock files
for signal_name in sorted(vol_stock_assign["signal_name"].unique()):
    for pf in PORTFOLIOS:
        sub = vol_stock_assign[
            (vol_stock_assign["signal_name"] == signal_name) &
            (vol_stock_assign["portfolio"] == pf)
        ]

        fname = f"volatility_model_{signal_name}_{pf.lower()}_stock_assignments.csv"
        path = os.path.join(OUTPUT_DIR, fname)
        sub.to_csv(path, index=False)

# Excel report
excel_path = os.path.join(OUTPUT_DIR, "volatility_model_report.xlsx")

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    vol_perf_summary.to_excel(writer, sheet_name="Perf Summary", index=False)
    vol_benchmark_summary.to_excel(writer, sheet_name="VS VNINDEX", index=False)
    vol_spread_summary.to_excel(writer, sheet_name="P1-P3 Summary", index=False)
    vol_spread.to_excel(writer, sheet_name="P1-P3 Returns", index=False)
    vol_cumulative.to_excel(writer, sheet_name="Cumulative", index=False)

    # Full stock assignment may be large; write latest date only to Excel.
    latest_assign.to_excel(writer, sheet_name="Latest Holdings", index=False)

# ZIP all outputs
zip_base = "volatility_model_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)

print("\nOUTPUT FILES CREATED")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {fname}")

print(f"\nZIP file created: {zip_path}")
_maybe_download(zip_path)

print("\n✓ Volatility model complete")
